# Attenuator Calibration Lab Script

Dirty science-lab notebook for attenuator calibration. It has two paths:

- embedded auto-calibration: run firmware, retrieve retained HAC3 records, and inspect them with helper plots;
- manual exploration: call `pcb.atten()`, wait, and read `pcb.pd()` so thresholds and failure modes can be changed quickly.

Measure dark in the preceding cell or in the dark cell below before collecting either dataset.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import math
import time
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares

import hispec_fibpcb as hspcb

BROKER = "hispec.caltech.edu"
LASER = "1028y"
OUTPUT = "yj_ao"
FIBER = "M"
PD_CHANNEL = "yj"

MANUAL_FIT_METHOD: Literal["firmware", "scipy"] = "firmware"
MANUAL_DWELL_S = 0.35
MANUAL_DWELL_MS = int(round(MANUAL_DWELL_S * 1000))
MANUAL_SWEEP_STEP_MV = 50.0
MANUAL_MAX_BRIDGES = 4
MANUAL_MAX_RECORDS = 160
MANUAL_SNR_USABLE = hspcb.ATTENUATOR_CAL_SNR_USABLE
MANUAL_DAC_SIGMA_MV = 3.0
TX_MIN = 1.0e-10
TX_MAX = 0.999999
MAX_DAC_MV = hspcb.ATTENUATOR_DRIVE_MAX_MV
ADC_CLIP_MV = hspcb.ATTENUATOR_ADC_CLIP_MV
GAIN = hspcb.ATTENUATOR_DEFAULT_GAIN

pcb = hspcb.HispecFibPcb(BROKER, connect=True)
pcb.status()


## Dark

Keep `persist=False` while exploring. Switch to `True` only after reviewing the dark value and noise. Duration captures return immediately, so wait before querying the resulting dark.


In [ ]:
pcb.set_laser_level(LASER, 0)
dark_duration_ms = 2000
pcb.pd_dark(PD_CHANNEL, duration_ms=dark_duration_ms, persist=False)
time.sleep(dark_duration_ms / 1000 + 0.25)
dark = pcb.pd_dark(PD_CHANNEL)
dark.dark


## Embedded Firmware Path

This is the authoritative firmware algorithm. It stores retained records in firmware and returns them through `atten_calibration_data()` after or during the run.


In [ ]:
embedded_status = pcb.atten_calibrate_auto(
    LASER,
    output=OUTPUT,
    fiber=FIBER,
    dwell_ms=MANUAL_DWELL_MS,
    persist=False,
)
embedded_status


In [ ]:
pcb.atten_calibration_status()


In [ ]:
embedded = pcb.atten_calibration_data(physical="all")
embedded


In [ ]:
embedded.bridge_table()


In [ ]:
embedded.plot_physical("dac1")
embedded.plot_physical("dac2")


In [ ]:
coeff = pcb.atten_coeff(LASER)
embedded.plot_surface(coeff.dac1, coeff.dac2, axis="fvoa_mv", overlay_records=True)


## Manual Exploration Path

This section intentionally reimplements acquisition in notebook code with direct `atten()`, wait, and `pd()` calls. It is for changing thresholds and probing failure modes quickly. Use `MANUAL_FIT_METHOD = "firmware"` to mirror the embedded weighted fit, or `"scipy"` for exploratory least-squares.


In [ ]:
@dataclass
class CalRecord:
    physical: str
    event: str
    sweep_mv: float
    other_mv: float
    mean_mv: float
    signal_mv: float
    rms_mv: float
    sigma_y_mv: float
    snr: float
    scale: float
    scale_sigma: float
    flux: float
    flux_sigma: float
    tx: float
    usable: bool
    saturated: bool
    fit_eligible: bool
    sigma_x_mv: float = MANUAL_DAC_SIGMA_MV
    included: bool = False
    b: float = np.nan
    residual_db: float = np.nan


def _set_pair(physical: str, sweep_mv: float, other_mv: float):
    sweep_mv = float(np.clip(sweep_mv, 0.0, MAX_DAC_MV))
    other_mv = float(np.clip(other_mv, 0.0, MAX_DAC_MV))
    if physical == "dac1":
        return pcb.atten(LASER, value1_mv=sweep_mv, value2_mv=other_mv)
    if physical == "dac2":
        return pcb.atten(LASER, value1_mv=other_mv, value2_mv=sweep_mv)
    raise ValueError("physical must be dac1 or dac2")


def _pd_window(channel: str = PD_CHANNEL):
    pd = pcb.pd(channel)
    pd_channel = getattr(pd, channel)
    if pd_channel is None:
        raise RuntimeError(f"missing photodiode channel {channel}")
    return pd_channel.window


def _normalize_fields(
    rec: CalRecord,
    *,
    open_signal_mv: float | None,
    open_sigma_mv: float,
    scale_rel_var: float,
) -> CalRecord:
    if not rec.usable or open_signal_mv is None or open_signal_mv <= 0.0 or rec.scale <= 0.0:
        return rec
    rec.flux = rec.signal_mv / rec.scale
    rel_var = (
        (rec.sigma_y_mv / rec.signal_mv) ** 2
        + scale_rel_var
        + (open_sigma_mv / open_signal_mv) ** 2
    )
    rec.flux_sigma = abs(rec.flux) * math.sqrt(max(rel_var, 0.0))
    rec.tx = rec.signal_mv / (open_signal_mv * rec.scale)
    rec.fit_eligible = TX_MIN < rec.tx < TX_MAX
    return rec


def measure_point(
    physical: str,
    sweep_mv: float,
    other_mv: float,
    *,
    event: str = "point",
    scale: float = 1.0,
    scale_rel_var: float = 0.0,
    open_signal_mv: float | None = None,
    open_sigma_mv: float = 0.0,
    fit_candidate: bool = False,
) -> CalRecord:
    _set_pair(physical, sweep_mv, other_mv)
    time.sleep(MANUAL_DWELL_S)
    window = _pd_window(PD_CHANNEL)
    signal_mv = float(window.mean_net_mv)
    mean_mv = float(window.mean_mv)
    rms_mv = float(window.rms_mv)
    sigma_y_mv = max(float(window.mean_net_err_mv), 1.0e-12)
    max_mv = float(window.max_mv)
    saturated = max_mv >= ADC_CLIP_MV
    snr = signal_mv / sigma_y_mv if signal_mv > 0.0 else -np.inf
    usable = bool((not saturated) and signal_mv > 0.0 and np.isfinite(snr) and snr >= MANUAL_SNR_USABLE)
    scale_sigma = abs(scale) * math.sqrt(max(scale_rel_var, 0.0))
    rec = CalRecord(
        physical=physical,
        event=event,
        sweep_mv=float(sweep_mv),
        other_mv=float(other_mv),
        mean_mv=mean_mv,
        signal_mv=signal_mv,
        rms_mv=rms_mv,
        sigma_y_mv=sigma_y_mv,
        snr=float(snr),
        scale=float(scale),
        scale_sigma=scale_sigma,
        flux=np.nan,
        flux_sigma=np.nan,
        tx=np.nan,
        usable=usable,
        saturated=bool(saturated),
        fit_eligible=False,
    )
    _normalize_fields(
        rec,
        open_signal_mv=open_signal_mv,
        open_sigma_mv=open_sigma_mv,
        scale_rel_var=scale_rel_var,
    )
    rec.fit_eligible = bool(fit_candidate and rec.fit_eligible)
    print(
        f"{event:14s} {physical}={sweep_mv:8.2f} other={other_mv:8.2f} "
        f"signal={signal_mv:9.3f}+/-{sigma_y_mv:6.3f} snr={snr:8.2f} "
        f"usable={usable} saturated={saturated} scale={scale:.6g}"
    )
    return rec


In [ ]:
def find_companion_start(physical: str, *, step_mv: float = 5.0) -> tuple[float, list[CalRecord]]:
    records: list[CalRecord] = []
    low, high = 0.0, MAX_DAC_MV
    for _ in range(16):
        mid = 0.5 * (low + high)
        rec = measure_point(physical, 0.0, mid, event="initial_probe")
        records.append(rec)
        if rec.saturated:
            low = mid
        else:
            high = mid
        if high - low <= step_mv:
            break
    return high, records


def bridge_once(
    physical: str,
    sweep_mv: float,
    other_mv: float,
    open_signal_mv: float,
    open_sigma_mv: float,
    scale: float,
    scale_rel_var: float,
    records: list[CalRecord],
    *,
    step_mv: float = 5.0,
) -> tuple[float, float, float, bool]:
    before = measure_point(
        physical,
        sweep_mv,
        other_mv,
        event="bridge_before",
        scale=scale,
        scale_rel_var=scale_rel_var,
        open_signal_mv=open_signal_mv,
        open_sigma_mv=open_sigma_mv,
        fit_candidate=True,
    )
    records.append(before)
    if not before.usable or before.signal_mv <= 0.0:
        return other_mv, scale, scale_rel_var, False

    low, high = 0.0, other_mv
    for _ in range(16):
        mid = 0.5 * (low + high)
        probe = measure_point(
            physical,
            sweep_mv,
            mid,
            event="bridge_probe",
            scale=scale,
            scale_rel_var=scale_rel_var,
            open_signal_mv=open_signal_mv,
            open_sigma_mv=open_sigma_mv,
        )
        records.append(probe)
        if probe.saturated:
            low = mid
        else:
            high = mid
        if high - low <= step_mv:
            break

    after = measure_point(
        physical,
        sweep_mv,
        high,
        event="bridge_after",
        scale=scale,
        scale_rel_var=scale_rel_var,
        open_signal_mv=open_signal_mv,
        open_sigma_mv=open_sigma_mv,
        fit_candidate=True,
    )
    records.append(after)
    if not after.usable or after.signal_mv <= before.signal_mv:
        return other_mv, scale, scale_rel_var, False

    ratio = after.signal_mv / before.signal_mv
    scale *= ratio
    scale_rel_var += (before.sigma_y_mv / before.signal_mv) ** 2 + (after.sigma_y_mv / after.signal_mv) ** 2
    after.scale = scale
    after.scale_sigma = abs(scale) * math.sqrt(max(scale_rel_var, 0.0))
    _normalize_fields(after, open_signal_mv=open_signal_mv, open_sigma_mv=open_sigma_mv, scale_rel_var=scale_rel_var)
    return high, scale, scale_rel_var, True


def acquire_physical(
    physical: str,
    *,
    step_mv: float = MANUAL_SWEEP_STEP_MV,
    max_records: int = MANUAL_MAX_RECORDS,
    max_bridges: int = MANUAL_MAX_BRIDGES,
) -> list[CalRecord]:
    other_mv, records = find_companion_start(physical)
    reference = measure_point(physical, 0.0, other_mv, event="reference", fit_candidate=True)
    records.append(reference)
    if not reference.usable:
        raise RuntimeError(f"{physical} open reference is not usable: {reference}")

    open_signal_mv = reference.signal_mv
    open_sigma_mv = reference.sigma_y_mv
    scale = 1.0
    scale_rel_var = 0.0
    sweep_mv = 0.0
    bridges = 0
    while sweep_mv <= MAX_DAC_MV + 0.5 * step_mv and len(records) < max_records:
        rec = measure_point(
            physical,
            min(sweep_mv, MAX_DAC_MV),
            other_mv,
            scale=scale,
            scale_rel_var=scale_rel_var,
            open_signal_mv=open_signal_mv,
            open_sigma_mv=open_sigma_mv,
            fit_candidate=True,
        )
        records.append(rec)
        if rec.usable:
            sweep_mv += step_mv
            continue
        last_usable_mv = max(0.0, sweep_mv - step_mv)
        if bridges >= max_bridges:
            break
        other_mv, scale, scale_rel_var, bridged = bridge_once(
            physical,
            last_usable_mv,
            other_mv,
            open_signal_mv,
            open_sigma_mv,
            scale,
            scale_rel_var,
            records,
        )
        bridges += 1
        if not bridged:
            break
        sweep_mv = last_usable_mv + step_mv
    return records


In [ ]:
def _fit_candidates(records: list[CalRecord]) -> list[CalRecord]:
    return [
        r for r in records
        if r.fit_eligible
        and r.usable
        and np.isfinite(r.tx)
        and TX_MIN < r.tx < TX_MAX
        and np.isfinite(r.flux)
        and r.flux > 0.0
        and np.isfinite(r.flux_sigma)
        and r.flux_sigma > 0.0
    ]


def _tx_to_b(tx):
    return hspcb._atten_model_b_from_tx(np.asarray(tx, dtype=float))


def _b_to_tx(b):
    return hspcb._atten_model_tx_from_b(np.asarray(b, dtype=float))


def _sigma_delta(record: CalRecord) -> float:
    sigma_tx = abs(record.tx) * (record.flux_sigma / record.flux)
    tx_lo = max(TX_MIN, record.tx - sigma_tx)
    tx_hi = min(TX_MAX, record.tx + sigma_tx)
    b_lo = float(_tx_to_b(tx_lo))
    b_hi = float(_tx_to_b(tx_hi))
    return max(abs(b_hi - b_lo) * 0.5, 1.0e-6)


def fit_records_firmware_style(records: list[CalRecord], *, gain: float = GAIN) -> dict[str, float | int | bool]:
    for r in records:
        r.included = False
        r.b = np.nan
        r.residual_db = np.nan
    fit_records = _fit_candidates(records)
    if len(fit_records) < 6:
        raise RuntimeError(f"not enough fit records: {len(fit_records)}")

    x = gain * np.array([r.sweep_mv for r in fit_records], dtype=float)
    y = _tx_to_b([r.tx for r in fit_records])
    sigma_delta = np.array([_sigma_delta(r) for r in fit_records], dtype=float)
    sigma_x = gain * np.array([r.sigma_x_mv for r in fit_records], dtype=float)
    slope = 0.0
    intercept = 0.0
    for _ in range(3):
        var = sigma_delta * sigma_delta + slope * slope * sigma_x * sigma_x
        w = 1.0 / np.maximum(var, 1.0e-12)
        sw = np.sum(w)
        sx = np.sum(w * x)
        sy = np.sum(w * y)
        sxx = np.sum(w * x * x)
        sxy = np.sum(w * x * y)
        denom = sw * sxx - sx * sx
        if not (denom > 0.0):
            raise RuntimeError("manual firmware-style fit is singular")
        slope = (sw * sxy - sx * sy) / denom
        intercept = (sy - slope * sx) / sw
        if not (slope > 0.0 and np.isfinite(intercept)):
            raise RuntimeError("manual firmware-style fit did not produce a positive slope")

    predicted_tx = _b_to_tx(slope * x + intercept)
    measured_tx = np.array([r.tx for r in fit_records], dtype=float)
    residual_db = 10.0 * np.log10(np.clip(predicted_tx, 1.0e-300, np.inf) / np.clip(measured_tx, 1.0e-300, np.inf))
    for r, b, residual in zip(fit_records, y, residual_db):
        r.included = True
        r.b = float(b)
        r.residual_db = float(residual)
    fvoa_50pct_mv = -intercept / slope
    return {
        "method": "firmware",
        "accepted": bool(fvoa_50pct_mv > 0.0 and slope > 0.0),
        "points": len(fit_records),
        "fvoa_50pct_mv": float(fvoa_50pct_mv),
        "slope_inv_fvoa_mv": float(slope),
        "gain": float(gain),
        "rms_db": float(np.sqrt(np.mean(residual_db * residual_db))),
        "max_abs_db": float(np.max(np.abs(residual_db))),
    }


def fit_records_scipy(records: list[CalRecord], *, gain: float = GAIN) -> dict[str, float | int | bool]:
    fit_records = _fit_candidates(records)
    if len(fit_records) < 6:
        raise RuntimeError(f"not enough fit records: {len(fit_records)}")
    x = gain * np.array([r.sweep_mv for r in fit_records], dtype=float)
    y = _tx_to_b([r.tx for r in fit_records])
    sigma_delta = np.array([_sigma_delta(r) for r in fit_records], dtype=float)

    firmware_guess = fit_records_firmware_style(records, gain=gain)
    initial = np.array([firmware_guess["fvoa_50pct_mv"], firmware_guess["slope_inv_fvoa_mv"]], dtype=float)

    def residual(params):
        f50, slope = params
        return (slope * (x - f50) - y) / np.maximum(sigma_delta, 1.0e-6)

    result = least_squares(residual, initial, bounds=([0.0, 0.0], [np.inf, np.inf]), loss="soft_l1")
    f50, slope = result.x
    predicted_tx = _b_to_tx(slope * (x - f50))
    measured_tx = np.array([r.tx for r in fit_records], dtype=float)
    residual_db = 10.0 * np.log10(np.clip(predicted_tx, 1.0e-300, np.inf) / np.clip(measured_tx, 1.0e-300, np.inf))
    for r, b, residual_value in zip(fit_records, y, residual_db):
        r.included = True
        r.b = float(b)
        r.residual_db = float(residual_value)
    return {
        "method": "scipy",
        "accepted": bool(result.success and f50 > 0.0 and slope > 0.0),
        "points": len(fit_records),
        "fvoa_50pct_mv": float(f50),
        "slope_inv_fvoa_mv": float(slope),
        "gain": float(gain),
        "rms_db": float(np.sqrt(np.mean(residual_db * residual_db))),
        "max_abs_db": float(np.max(np.abs(residual_db))),
    }


def fit_records(records: list[CalRecord], *, method: str = MANUAL_FIT_METHOD, gain: float = GAIN):
    if method == "firmware":
        return fit_records_firmware_style(records, gain=gain)
    if method == "scipy":
        return fit_records_scipy(records, gain=gain)
    raise ValueError("method must be 'firmware' or 'scipy'")


In [ ]:
def plot_manual_records(records: list[CalRecord], fit: dict[str, float | int | bool]):
    sweep = np.array([r.sweep_mv for r in records], dtype=float)
    fvoa = sweep * float(fit["gain"])
    signal = np.array([r.signal_mv for r in records], dtype=float)
    sigma_y = np.array([r.sigma_y_mv for r in records], dtype=float)
    tx = np.array([r.tx for r in records], dtype=float)
    flux = np.array([r.flux for r in records], dtype=float)
    flux_sigma = np.array([r.flux_sigma for r in records], dtype=float)
    included = np.array([r.included for r in records], dtype=bool)
    usable = np.array([r.usable for r in records], dtype=bool)
    residual = np.array([r.residual_db for r in records], dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        attenuation_db = hspcb._atten_db_from_tx(tx)
        attenuation_sigma_db = (10.0 / math.log(10.0)) * (flux_sigma / flux)

    fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True, constrained_layout=True)
    axes[0].errorbar(fvoa[usable], signal[usable], yerr=sigma_y[usable], fmt=".", color="tab:blue", ecolor="0.6", label="usable")
    axes[0].scatter(fvoa[~usable], signal[~usable], marker="x", color="tab:red", label="not usable")
    axes[0].set_ylabel("signal_mv")
    axes[0].legend(loc="best")

    tx_mask = np.isfinite(attenuation_db) & np.isfinite(attenuation_sigma_db)
    axes[1].errorbar(fvoa[tx_mask], attenuation_db[tx_mask], yerr=attenuation_sigma_db[tx_mask], fmt=".", color="0.25", ecolor="0.65")
    grid = np.linspace(0.0, MAX_DAC_MV * float(fit["gain"]), 400)
    b = float(fit["slope_inv_fvoa_mv"]) * (grid - float(fit["fvoa_50pct_mv"]))
    axes[1].plot(grid, hspcb._atten_db_from_tx(hspcb._atten_model_tx_from_b(b)), color="tab:orange", label=f"{fit['method']} fit")
    axes[1].set_ylabel("attenuation_db")
    axes[1].legend(loc="best")

    axes[2].axhline(0.0, color="0.35", linestyle="--", linewidth=0.8)
    axes[2].scatter(fvoa[included], residual[included], color="black", s=22)
    axes[2].set_ylabel("residual_db")
    axes[2].set_xlabel("swept FVOA drive (mV)")
    return fig


In [ ]:
pcb.set_laser_level(LASER, 50)
dac1_records = acquire_physical("dac1", step_mv=MANUAL_SWEEP_STEP_MV)
dac1_fit = fit_records(dac1_records, method=MANUAL_FIT_METHOD)
dac1_fit


In [ ]:
plot_manual_records(dac1_records, dac1_fit)


In [ ]:
dac2_records = acquire_physical("dac2", step_mv=MANUAL_SWEEP_STEP_MV)
dac2_fit = fit_records(dac2_records, method=MANUAL_FIT_METHOD)
dac2_fit


In [ ]:
plot_manual_records(dac2_records, dac2_fit)


In [ ]:
manual_records = [asdict(r) for r in dac1_records + dac2_records]
manual_coeff = {
    "dac1": {k: dac1_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
    "dac2": {k: dac2_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
}
manual_coeff


In [ ]:
# Review plots before applying. Keep persist=False until repeated and accepted.
# pcb.set_atten_coeff(LASER, manual_coeff["dac1"], manual_coeff["dac2"], persist=False)
